# Análisis de Datos · Semana 9
## Acumuladores, banderas y ciclos anidados

**TIA502 · Facultad de Empresariales · Profesor David Escobar-Castillejos**

Tres patrones que resuelven casi todo lo que se hace dentro de un ciclo. No son tres temas: son
tres formas de la misma estructura, y reconocer cuál pide el enunciado es la mitad de resolverlo.

Vale la pena decir a dónde va esto. **Todo lo de hoy reaparece en la semana 15 como una sola línea
de pandas.** El acumulador se vuelve `.sum()`, el contador se vuelve `.count()`, la bandera se
vuelve `.any()` y el ciclo anidado se vuelve `groupby` con dos columnas. Practicarlo a mano ahora
es lo que después te deja confiar en la línea corta.

Al terminar este cuaderno vas a poder:

1. Escribir un acumulador, un contador y una bandera, y decir qué pregunta contesta cada uno.
2. Interrumpir o saltar una vuelta con `break` y `continue`.
3. Usar el `else` del `for`, que casi nadie conoce.
4. Leer un ciclo anidado y predecir cuántas vueltas da antes de ejecutarlo.

### Cómo se usa este cuaderno

Ejecuta las celdas en orden. Tres fallan a propósito o dan un resultado inesperado a propósito.

Los datos son las cuatro campañas de la semana pasada.

In [ ]:
campanas = ["Instagram", "Meta", "Google", "TikTok"]
clics = [5074, 3820, 6910, 1240]
inversion = [38500, 29800, 51200, 9600]
conversiones = [173, 118, 241, 39]

print(f"{'Campaña':<12}{'Clics':>8}{'Inversión':>12}{'CPC':>8}")
for i in range(len(campanas)):
    print(f"{campanas[i]:<12}{clics[i]:>8,}{inversion[i]:>12,}{inversion[i] / clics[i]:>8.2f}")

---
# Bloque 1 · Los tres patrones

Acumular, contar y marcar.

| Patrón | La pregunta | Valor inicial | Dentro del ciclo |
|---|---|---|---|
| Acumulador | ¿Cuánto suman? | `0` | `total += valor` |
| Contador | ¿Cuántos cumplen? | `0` | `if condición: n += 1` |
| Bandera | ¿Existe al menos uno? | `False` | `if condición: hallado = True` |

Los tres comparten la misma estructura. **Se declara una variable antes del ciclo**, con un valor
inicial que representa "todavía nada". **Dentro del ciclo se actualiza** en cada vuelta. **Al
terminar**, esa variable tiene la respuesta.

In [ ]:
total_inversion = 0
campanas_grandes = 0
hay_cara = False

for i in range(len(campanas)):
    total_inversion += inversion[i]

    if clics[i] > 5000:
        campanas_grandes += 1

    if inversion[i] / clics[i] > 7.75:
        hay_cara = True

print(f"Inversión total:  ${total_inversion:,}")
print(f"Campañas grandes: {campanas_grandes}")
print(f"Hay alguna cara:  {hay_cara}")

Un solo recorrido, tres respuestas.

**Las tres variables se declaran antes del `for`**, y por eso sobreviven a todas las vueltas.

**El acumulador** suma cuánto. Contesta una pregunta de magnitud.

**El contador** suma uno. Contesta una pregunta de cuántos, no de cuánto.

**La bandera** solo pasa de `False` a `True`, y ya nunca regresa. Contesta si existe alguno.

## Contar y sumar no son lo mismo

Este es el error de lectura que más aparece en un parcial. Léelo con cuidado: "cuántas campañas
superan la meta" y "cuánto suman las campañas que superan la meta" son dos preguntas.

In [ ]:
META = 0.03

cuantas = 0
cuanto = 0

for i in range(len(campanas)):
    conversion = conversiones[i] / clics[i]
    if conversion >= META:
        cuantas += 1
        cuanto += inversion[i]

print(f"¿Cuántas superan la meta?      {cuantas}")
print(f"¿Cuánto suman esas campañas?  ${cuanto:,}")

Tres campañas y 119 500 pesos. Los dos números salen del mismo recorrido y contestan cosas
distintas.

La señal en el enunciado: **"cuántos" es un contador, "cuánto" es un acumulador.** Casi siempre
está en la primera palabra de la pregunta.

## El error clásico: declararla adentro

**Predice antes de correr.** ¿Cuánto vale `total` al terminar?

- **A.** 600, porque suma las tres.
- **B.** 300, porque se reinicia en cada vuelta.
- **C.** 0, porque el total se declara al final.
- **D.** Un error, porque `total` no existe antes del ciclo.

In [ ]:
# FALLA A PROPÓSITO. El acumulador está dentro del ciclo.
ventas = [100, 200, 300]

for v in ventas:
    total = 0
    total += v

print(total)

La respuesta es **B**, 300. Cada vuelta borra el total y vuelve a empezar, así que al final vale lo
del último registro.

El programa corre, no marca error, y el resultado equivocado se ve perfectamente normal. Trescientos
es un número creíble para una suma de tres ventas.

**La sangría es lo que decide cuál es cuál.** Compara las dos versiones.

In [ ]:
print("Con el acumulador DENTRO:")
for v in ventas:
    total = 0
    total += v
    print(f"  vuelta con v={v}, total queda en {total}")
print("  final:", total)

print()
print("Con el acumulador FUERA:")
total = 0
for v in ventas:
    total += v
    print(f"  vuelta con v={v}, total queda en {total}")
print("  final:", total)

El rastro vuelta por vuelta lo hace evidente. Cuando un total no cuadre, imprimirlo dentro del
ciclo es la forma más rápida de ver dónde se pierde.

## Acumular el máximo y el mínimo

Hay una cuarta forma del mismo patrón, y es la que sirve para "el mejor" y "el peor".

In [ ]:
mejor_i = 0
peor_i = 0

for i in range(len(campanas)):
    if inversion[i] / clics[i] < inversion[mejor_i] / clics[mejor_i]:
        mejor_i = i
    if inversion[i] / clics[i] > inversion[peor_i] / clics[peor_i]:
        peor_i = i

print(f"Mejor CPC: {campanas[mejor_i]:<12} ${inversion[mejor_i] / clics[mejor_i]:.2f}")
print(f"Peor CPC:  {campanas[peor_i]:<12} ${inversion[peor_i] / clics[peor_i]:.2f}")

El valor inicial no es cero: es **el primer elemento**. Empezar en cero rompería la búsqueda del
mínimo, porque ningún costo por clic es menor que cero y el resultado sería siempre el cero
inicial.

Es el error de "valor inicial imposible", y aparece cada vez que alguien inicializa un mínimo en
cero.

In [ ]:
# FALLA A PROPÓSITO. Inicializar un mínimo en cero.
minimo_mal = 0

for i in range(len(campanas)):
    cpc = inversion[i] / clics[i]
    if cpc < minimo_mal:
        minimo_mal = cpc

print("Mínimo empezando en cero:", minimo_mal, "<- ninguna campaña es más barata que gratis")

---
# Bloque 2 · Romper el flujo

Tres instrucciones que cambian el recorrido normal de un ciclo. La tercera casi nadie la conoce.

| Instrucción | Qué hace | Cuándo se usa |
|---|---|---|
| `break` | Sale del ciclo de inmediato | Ya encontraste lo que buscabas y seguir es desperdicio |
| `continue` | Salta a la siguiente vuelta | Este registro no aplica y no quieres anidar un `if` enorme |
| `else` del `for` | Corre solo si el ciclo terminó sin `break` | Para decir "recorrí todo y no encontré nada" |

In [ ]:
for i in range(len(campanas)):
    if clics[i] < 2000:
        continue

    if inversion[i] / clics[i] > 7.75:
        print(f"Primera cara: {campanas[i]}")
        break
else:
    print("Ninguna campaña rebasa el umbral.")

**`continue`.** Las campañas con menos de 2 000 clics no tienen suficiente volumen para juzgarlas.
Se saltan sin anidar un `if` alrededor de todo lo demás.

**`break`.** En cuanto encuentra la primera, sale. Recorrer las demás no cambiaría la respuesta.

**El `else`.** Va alineado con el `for`, no con el `if`. Corre solo si el ciclo llegó al final sin
toparse con un `break`.

La traza:

| `i` | Campaña | clics | Qué hace |
|---|---|---|---|
| 0 | Instagram | 5074 | Pasa el filtro. 7.59 no rebasa 7.75, sigue |
| 1 | Meta | 3820 | Pasa el filtro. 7.80 sí rebasa, imprime y sale |
| 2 | Google | 6910 | No se evalúa, el `break` ya salió |
| 3 | TikTok | 1240 | No se evalúa |

El `else` del `for` no corre, porque el ciclo salió por `break`. Ese es exactamente su propósito.

Súbele el umbral para ver el otro caso.

In [ ]:
UMBRAL = 20.00

for i in range(len(campanas)):
    if clics[i] < 2000:
        continue

    if inversion[i] / clics[i] > UMBRAL:
        print(f"Primera cara: {campanas[i]}")
        break
else:
    print(f"Ninguna campaña rebasa los ${UMBRAL:.2f} por clic.")

Ahora sí corrió el `else`, porque el ciclo llegó al final sin `break`.

Sin ese `else`, para conseguir lo mismo harías falta una bandera y un `if` después del ciclo. El
`else` del `for` es exactamente eso, empaquetado.

In [ ]:
# Lo mismo, con bandera. Funciona igual y ocupa dos líneas más.
encontrada = False

for i in range(len(campanas)):
    if clics[i] < 2000:
        continue
    if inversion[i] / clics[i] > UMBRAL:
        print(f"Primera cara: {campanas[i]}")
        encontrada = True
        break

if not encontrada:
    print(f"Ninguna campaña rebasa los ${UMBRAL:.2f} por clic.")

Las dos formas son correctas. La de la bandera se entiende sin conocer el `else` del `for`, y por
eso mucha gente la prefiere. La otra es más corta y no puede olvidársele actualizar la bandera.

## El riesgo del `else` mal alineado

In [ ]:
# FALLA A PROPÓSITO. El else está alineado con el if, no con el for.
for i in range(len(campanas)):
    if inversion[i] / clics[i] > 7.75:
        print(f"Cara: {campanas[i]}")
    else:
        print(f"  (barata: {campanas[i]})")

Ese `else` pertenece al `if` y corre en cada vuelta que no cumple, que es una cosa completamente
distinta. Los dos programas son válidos y hacen cosas diferentes, y lo único que los separa son
cuatro espacios.

---
# Bloque 3 · Ciclos anidados

Un ciclo dentro de otro. El de adentro da todas sus vueltas por cada vuelta del de afuera.

In [ ]:
regiones = ["Norte", "Centro"]
canales = ["Retail", "Online"]

for region in regiones:
    for canal in canales:
        print(f"{region} · {canal}")

**La multiplicación.** Dos regiones por dos canales dan cuatro vueltas. Con cuatro y tres serían
doce.

**El orden.** El de afuera avanza una posición solo cuando el de adentro terminó todas las suyas.

**El límite.** Cien por cien son diez mil vueltas. Anidar tres niveles sobre listas largas se vuelve
lento rápido.

Cuéntalas en lugar de suponerlas.

In [ ]:
regiones = ["Norte", "Centro", "Sur", "Oeste"]
canales = ["Retail", "Online", "Mayoreo"]

vueltas = 0
for region in regiones:
    for canal in canales:
        vueltas += 1

print(f"{len(regiones)} regiones × {len(canales)} canales = {vueltas} vueltas")

## Un cruce con datos

El anidado se vuelve útil cuando cada combinación produce un renglón de reporte.

In [ ]:
# Una cifra por combinación, escrita a mano para el ejemplo.
VENTAS = {
    ("Norte", "Retail"): 1331426, ("Norte", "Online"): 978286, ("Norte", "Mayoreo"): 2042264,
    ("Centro", "Retail"): 490472, ("Centro", "Online"): 1291740, ("Centro", "Mayoreo"): 2136767,
    ("Sur", "Retail"): 271090, ("Sur", "Online"): 420216, ("Sur", "Mayoreo"): 861697,
    ("Oeste", "Retail"): 738049, ("Oeste", "Online"): 589081, ("Oeste", "Mayoreo"): 1702889,
}

print(f"{'Región':<10}" + "".join(f"{c:>12}" for c in canales) + f"{'Total':>12}")
print("-" * 58)

gran_total = 0
for region in regiones:
    fila = 0
    for canal in canales:
        fila += VENTAS[(region, canal)]
    gran_total += fila
    celdas = "".join(f"{VENTAS[(region, canal)] / 1000:>12,.0f}" for canal in canales)
    print(f"{region:<10}{celdas}{fila / 1000:>12,.0f}")

print("-" * 58)
print(f"{'Total':<10}{'':>36}{gran_total / 1000:>12,.0f}")

Un acumulador por fila y otro global, dentro de un anidado. Esas veinte líneas son exactamente lo
que la semana 15.3 escribe así:

```python
ventas.pivot_table(index="region", columns="channel", values="amount",
                   aggfunc="sum", margins=True)
```

Vale la pena verlas juntas una vez. Lo que pandas te quita no es el concepto, es la contabilidad.

## Reusar la variable del ciclo interno

In [ ]:
# FALLA A PROPÓSITO. Los dos ciclos usan i.
for i in range(3):
    for i in range(2):        # el de adentro pisa al de afuera
        pass
    print("Vuelta de afuera, i vale:", i)

El `i` de afuera se perdió: al terminar el interno vale 1, siempre. Con listas cortas el síntoma es
sutil; con índices de verdad, el recorrido se descompone en silencio.

**Las variables de los dos ciclos tienen que llamarse distinto**, y de preferencia decir qué
recorren: `region` y `canal` en lugar de `i` y `j`.

## ¿Hacía falta anidar?

Antes de meter un ciclo dentro de otro, pregunta si un `continue` en el primero resolvía lo mismo.

In [ ]:
# Anidado innecesario: recorrer todo y filtrar dentro.
print("Con anidado:")
for region in regiones:
    for canal in canales:
        if canal == "Mayoreo":
            print(f"  {region} · {canal}: {VENTAS[(region, canal)] / 1000:,.0f}")

# Lo mismo, sin el ciclo interno.
print()
print("Sin anidado:")
for region in regiones:
    print(f"  {region} · Mayoreo: {VENTAS[(region, 'Mayoreo')] / 1000:,.0f}")

Doce vueltas contra cuatro, para el mismo resultado. El ciclo interno existía solo para descartar
dos de cada tres casos.

---
# Ejercicios

Las soluciones están hasta abajo del cuaderno.

## Los tres patrones

### Ejercicio 1 · Uno de cada uno

En un solo recorrido de las cuatro campañas, calcula: el total de clics, cuántas tienen más de
150 conversiones, y si existe alguna con conversión por debajo del 3 %.

### Ejercicio 2 · Contar contra sumar

Contesta las cuatro preguntas, distinguiendo bien cuál es contador y cuál acumulador:

1. ¿Cuántas campañas cuestan más de 7.60 por clic?
2. ¿Cuánto se invirtió en esas campañas?
3. ¿Cuántas conversiones trajeron entre todas?
4. ¿Cuánto costó cada conversión, en promedio, en esas campañas?

### Ejercicio 3 · El mejor por otra métrica

Encuentra la campaña con el mejor costo por conversión, no por clic. Imprime su nombre, su costo
por conversión y cuánto mejor es que el peor.

Inicializa con el primer elemento, no con cero.

## Romper el flujo

### Ejercicio 4 · `continue` que evita un `if` grande

Escribe un ciclo que reporte el costo por conversión solo de las campañas con más de 100
conversiones, usando `continue` para saltar las demás.

Después escríbelo con un `if` que envuelva todo el cuerpo, y compara cuál se lee mejor.

### Ejercicio 5 · `break` con `else`

Busca la primera campaña que cumpla dos condiciones a la vez: más de 5 000 clics y conversión por
arriba del 3.4 %. Si no hay ninguna, dilo con el `else` del `for`.

Prueba con un umbral que sí encuentre y con uno que no.

### Ejercicio 6 · Contar sin recorrer de más

Escribe un ciclo que se detenga en cuanto la inversión acumulada rebase 100 000, e imprima cuántas
campañas hicieron falta.

## Anidados

### Ejercicio 7 · Predecir las vueltas

Sin correr nada, di cuántas veces se imprime algo en cada uno de estos tres, y después compruébalo.

```python
for a in range(3):
    for b in range(4):
        print(a, b)

for a in range(3):
    for b in range(a):
        print(a, b)

for a in range(3):
    for b in range(4):
        if b == 2:
            break
        print(a, b)
```

El segundo y el tercero son los interesantes. Explica por qué en un comentario.

### Ejercicio 8 · Un reporte cruzado

Con dos listas de categorías de tu área, por ejemplo región y producto, escribe un ciclo anidado
que imprima cada combinación con una métrica calculada. Agrega un acumulador y un contador al
recorrido.

Las variables de los dos ciclos tienen que llamarse distinto y decir qué recorren.

La prueba: cuenta a mano cuántas líneas debería imprimir. Si no coincide, el anidado está mal.

---
## Tres ideas para llevarse

**La variable vive fuera del ciclo.** Declararla adentro la reinicia en cada vuelta, y el resultado
equivocado se ve perfectamente normal.

**Contar y sumar no son lo mismo.** "Cuántos cumplen" es un contador, "cuánto suman" es un
acumulador, y el enunciado casi siempre lo dice en la primera palabra.

**Las vueltas se multiplican.** Dos por dos son cuatro, y cien por cien son diez mil. Ahí es donde un
anidado deja de ser gratis.

La siguiente sesión es cómo empaquetar un cálculo para no volver a escribirlo nunca.

---
# Soluciones

### Ejercicio 1

```python
total_clics = 0
muchas_conversiones = 0
hay_baja = False

for i in range(len(campanas)):
    total_clics += clics[i]
    if conversiones[i] > 150:
        muchas_conversiones += 1
    if conversiones[i] / clics[i] < 0.03:
        hay_baja = True

print(f"Total de clics:            {total_clics:,}")
print(f"Con más de 150 conversiones: {muchas_conversiones}")
print(f"Hay alguna bajo el 3 %:     {hay_baja}")
```

Un solo recorrido para tres respuestas. Recorrer tres veces también funciona y cuesta tres veces
más, lo cual con cuatro campañas da igual y con trescientas mil no.

### Ejercicio 2

```python
UMBRAL = 7.60

cuantas = 0
invertido = 0
conv_totales = 0

for i in range(len(campanas)):
    if inversion[i] / clics[i] > UMBRAL:
        cuantas += 1
        invertido += inversion[i]
        conv_totales += conversiones[i]

print(f"1. Cuántas cuestan más de {UMBRAL}: {cuantas}")
print(f"2. Cuánto se invirtió:            ${invertido:,}")
print(f"3. Cuántas conversiones trajeron:  {conv_totales}")
print(f"4. Costo por conversión promedio: ${invertido / conv_totales:,.2f}")
```

La cuarta es la interesante: es el acumulado entre el acumulado, no el promedio de los promedios.
Es el mismo cuidado del costo por clic global de la semana pasada.

### Ejercicio 3

```python
mejor = 0
peor = 0

for i in range(len(campanas)):
    cpa = inversion[i] / conversiones[i]
    if cpa < inversion[mejor] / conversiones[mejor]:
        mejor = i
    if cpa > inversion[peor] / conversiones[peor]:
        peor = i

cpa_mejor = inversion[mejor] / conversiones[mejor]
cpa_peor = inversion[peor] / conversiones[peor]

print(f"Mejor: {campanas[mejor]:<12} ${cpa_mejor:>8,.2f} por conversión")
print(f"Peor:  {campanas[peor]:<12} ${cpa_peor:>8,.2f} por conversión")
print(f"El mejor cuesta {cpa_peor / cpa_mejor:.2f} veces menos que el peor")
```

Inicializar en `0` como **índice** es correcto; inicializar el costo en `0` como **valor** no lo
sería. La diferencia está en que aquí el cero es una posición válida de la lista, no un valor
imposible de la métrica.

### Ejercicio 4

```python
print("Con continue:")
for i in range(len(campanas)):
    if conversiones[i] <= 100:
        continue
    print(f"  {campanas[i]:<12} ${inversion[i] / conversiones[i]:>8,.2f}")

print("\nCon if envolvente:")
for i in range(len(campanas)):
    if conversiones[i] > 100:
        print(f"  {campanas[i]:<12} ${inversion[i] / conversiones[i]:>8,.2f}")
```

Con una sola línea dentro no se nota la diferencia. Se nota cuando el cuerpo tiene quince líneas:
el `continue` las deja todas a un nivel de sangría, y el `if` envolvente las mete todas un nivel
más adentro.

### Ejercicio 5

```python
for umbral_conv in [0.034, 0.10]:
    print(f"Buscando conversión sobre {umbral_conv:.1%}:")
    for i in range(len(campanas)):
        if clics[i] <= 5000:
            continue
        if conversiones[i] / clics[i] > umbral_conv:
            print(f"  Encontrada: {campanas[i]}")
            break
    else:
        print("  Ninguna cumple las dos condiciones.")
    print()
```

Con 3.4 % encuentra Google; con 10 % no encuentra nada y corre el `else`. Meter los dos umbrales en
un ciclo de afuera es lo que permite probar los dos caminos sin duplicar el código.

### Ejercicio 6

```python
LIMITE = 100000
acumulado = 0
cuantas = 0

for i in range(len(campanas)):
    acumulado += inversion[i]
    cuantas += 1
    if acumulado > LIMITE:
        break

print(f"Hicieron falta {cuantas} campañas para rebasar {LIMITE:,}")
print(f"Acumulado al detenerse: {acumulado:,}")
```

Tres campañas y 119 500. Nota que el contador sube **antes** del `break`, porque la campaña que
rebasó el límite sí cuenta. Ponerlo después daría dos, y las dos lecturas necesitan que decidas
cuál querías.

### Ejercicio 7

```python
# El primero: 12 veces. Tres por cuatro, sin condiciones.
# El segundo: 3 veces. range(a) da cero vueltas cuando a es 0, una cuando es 1
#   y dos cuando es 2. Cero más uno más dos son tres.
# El tercero: 6 veces. El break corta el ciclo interno en b == 2, así que cada
#   vuelta de afuera imprime b = 0 y b = 1, y son tres vueltas de afuera.

n = 0
for a in range(3):
    for b in range(4):
        n += 1
print("Primero:", n)

n = 0
for a in range(3):
    for b in range(a):
        n += 1
print("Segundo:", n)

n = 0
for a in range(3):
    for b in range(4):
        if b == 2:
            break
        n += 1
print("Tercero:", n)
```

El segundo es el patrón de "cada uno contra los anteriores", y aparece cuando comparas todos los
pares de una lista sin repetir. El tercero enseña que un `break` en el ciclo interno **solo sale del
interno**, no de los dos.

### Ejercicio 8

No hay solución publicada porque las categorías son distintas para cada quien. Se califica sobre
cuatro cosas: que las variables de los dos ciclos tengan nombres que digan qué recorren, que el
acumulador y el contador estén declarados fuera de los dos ciclos, que la cuenta a mano de los
renglones coincida con la salida, y que el reporte tenga encabezado.